# ScreenClean job runner

Runs the next job from `jobs/queue/` in the repo. No logins or keys are needed except
Google Drive access.

1. **Runtime → Change runtime type** → pick **T4 GPU** or **CPU**, whichever the job needs.
2. **Runtime → Run all**, and allow Google Drive access.
3. Keep this tab open until cell 3 prints `JOB FINISHED`, `TIME BUDGET REACHED` or `JOB FAILED`.
4. Cell 4 downloads `<job>.zip`. Move it from Downloads into your `results_inbox` folder.

**Is it still running?** The page can stop showing new lines while the job keeps working. Open
Google Drive → `screenclean/job_status/<job>.json`: its `progress` and `progress_utc` fields update
after every step.


In [ ]:
# Cell 1 — Settings (Colab form fields)
GH_USER = "ananya-baweja"   #@param {type:"string"}
REPO = "screenclean"        #@param {type:"string"}
BRANCH = "main"             #@param {type:"string"}
JOB = "auto"                #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/screenclean"  #@param {type:"string"}


In [ ]:
# Cell 2 — Setup (public repo, no login)
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
repo_dir = f"/content/{REPO}"
if os.path.isdir(repo_dir):
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    url = f"https://github.com/{GH_USER}/{REPO}.git"
    subprocess.run(["git", "clone", "-b", BRANCH, url, repo_dir], check=True)
%cd {repo_dir}
!git log --oneline -1
!pip install -q -e ".[colab]"


In [ ]:
# Cell 3 — Run the next job (prints JOB FINISHED / TIME BUDGET REACHED / JOB FAILED)
!python -m screenclean jobs run --job "{JOB}" --drive-root "{DRIVE_ROOT}"


In [ ]:
# Cell 4 — Download the results zip, then move it into your results_inbox folder
import pathlib
from google.colab import files
marker = pathlib.Path("/content/last_results_zip.txt")
if marker.exists():
    files.download(marker.read_text().strip())
    print("Downloaded. Move the zip from Downloads into results_inbox.")
    print("If nothing downloaded, the zip is also in Google Drive: screenclean/results_zips/")
else:
    print("No results zip this time. If it said TIME BUDGET REACHED, run the notebook again later.")
